# Python Foundations for ROS 2 Node Development

This notebook introduces Python through examples that directly support later ROS 2 work.

The main progression is:

1. Python variables and lists
2. Indexing and slicing
3. `for` loops and `range`
4. Files and directories with the `os` standard library
5. Finding ROS bag directories
6. NumPy arrays for numerical data
7. ROS 2–style message and node data patterns

The central distinction is:

- A **Python list** is a general-purpose container.
- A **NumPy array** is intended for numerical and matrix-oriented computation.
- ROS 2 messages often arrive as Python objects whose fields may contain numbers, strings, or lists.

## 1. Variables and Python lists

A Python list can contain values of different types. This makes it similar to a MATLAB cell array, although Python lists are used much more broadly.

In [ ]:
student_number = 3

data = [1, "Peter", student_number, -1, 3.14, -3.27]

print(data)
print(type(data))

Each element has an index. Python indexing begins at zero.

Negative indices count backward from the end.

In [ ]:
print("First element:", data[0])
print("Second element:", data[1])
print("Last element:", data[-1])

print("Type of second element:", type(data[1]))

## 2. List slicing

The slicing expression

```python
data[start:stop]
```

includes `start` but excludes `stop`.

This is why `data[2:4]` returns elements at indices 2 and 3.

In [ ]:
print(data[2:4])
print(data[2:5])

A useful memory rule is:

```text
start <= selected index < stop
```

In [ ]:
numbers = [10, 20, 30, 40, 50, 60]

print(numbers[1:4])
print(numbers[:3])
print(numbers[3:])
print(numbers[::2])

## 3. Adding, changing, and removing list elements

In [ ]:
names = ["motor", "encoder", "controller"]

names.append("battery")
print(names)

names[1] = "wheel_encoder"
print(names)

removed_item = names.pop()
print("Removed:", removed_item)
print("Remaining:", names)

## 4. Iterating directly through a list

When the index is not needed, iterate directly through the elements.

In [ ]:
sensor_names = ["imu", "lidar", "camera"]

for sensor_name in sensor_names:
    print("Sensor:", sensor_name)

This is usually clearer than manually indexing every element.

## 5. Using `range`

`range` produces integer sequences. It is commonly used when an index or repeated count is required.

In [ ]:
for index in range(5):
    print(index)

The stop value is excluded, just like list slicing.

In [ ]:
print(list(range(5)))
print(list(range(2, 6)))
print(list(range(0, 10, 2)))

## 6. Looping with both index and value

`enumerate` is usually better than writing `range(len(...))`.

In [ ]:
topics = ["/cmd_vel", "/odom", "/scan"]

for index, topic in enumerate(topics):
    print(index, topic)

The following also works, but it is less direct:

In [ ]:
for index in range(len(topics)):
    print(index, topics[index])

Use `range(len(...))` when you genuinely need index-based access or want to compare neighboring elements. Otherwise, prefer direct iteration or `enumerate`.

## 7. Conditional control flow

ROS 2 nodes often inspect message values, states, topic names, or file names before deciding what to do.

In [ ]:
battery_voltage = 11.4

if battery_voltage < 10.5:
    print("Critical battery")
elif battery_voltage < 11.5:
    print("Battery warning")
else:
    print("Battery normal")

## 8. Combining loops and conditions

In [ ]:
topic_names = ["/cmd_vel", "/odom", "/scan", "/camera/image_raw"]

for topic_name in topic_names:
    if "camera" in topic_name:
        print("Image topic:", topic_name)
    else:
        print("Non-image topic:", topic_name)

## 9. The `os` standard library

The `os` module provides operating-system services such as:

- reading the current directory,
- listing files,
- creating directories,
- joining paths,
- checking whether a path is a file or directory.

These operations are directly useful for ROS bag management, launch files, configuration folders, and logged experiment data.

In [ ]:
import os

current_directory = os.getcwd()

print("Current directory:")
print(current_directory)

## 10. Listing directory contents

In [ ]:
entries = os.listdir(current_directory)

print("Number of entries:", len(entries))

for entry in entries:
    print(entry)

`os.listdir()` returns a Python list of names.

This is a natural example of why Python lists matter in robotics software: operating-system functions frequently return collections of files and directories.

## 11. Building paths correctly

Avoid manually concatenating paths with `/` or `\`.

Use `os.path.join()` so the code works correctly across operating systems.

In [ ]:
example_folder = os.path.join(current_directory, "rosbag_data")
example_file = os.path.join(example_folder, "metadata.yaml")

print(example_folder)
print(example_file)

## 12. Separating files and directories

In [ ]:
files = []
directories = []

for entry in os.listdir(current_directory):
    full_path = os.path.join(current_directory, entry)

    if os.path.isfile(full_path):
        files.append(entry)
    elif os.path.isdir(full_path):
        directories.append(entry)

print("Files:", files)
print("Directories:", directories)

This pattern uses:

- one list for files,
- one list for directories,
- a `for` loop,
- an `if` statement,
- `append()` to build the result.

## 13. Creating a sample ROS bag directory structure

This notebook creates a harmless local teaching example. It does not require ROS 2 to be installed.

In [ ]:
teaching_root = os.path.join(current_directory, "teaching_rosbags")
os.makedirs(teaching_root, exist_ok=True)

sample_bags = [
    "rosbag2_2026_07_20-10_00_00",
    "rosbag2_2026_07_21-14_30_00",
    "notes",
]

for folder_name in sample_bags:
    folder_path = os.path.join(teaching_root, folder_name)
    os.makedirs(folder_path, exist_ok=True)

print(os.listdir(teaching_root))

## 14. Finding ROS bag directories

ROS 2 bag folders commonly use names beginning with `rosbag2_`.

We can filter directory names using a loop and a condition.

In [ ]:
rosbag_directories = []

for entry in os.listdir(teaching_root):
    full_path = os.path.join(teaching_root, entry)

    if os.path.isdir(full_path) and entry.startswith("rosbag2_"):
        rosbag_directories.append(entry)

print("ROS bag directories:")
for bag_directory in rosbag_directories:
    print(bag_directory)

A shorter list-comprehension form is also possible:

In [ ]:
rosbag_directories_compact = [
    entry
    for entry in os.listdir(teaching_root)
    if os.path.isdir(os.path.join(teaching_root, entry))
    and entry.startswith("rosbag2_")
]

print(rosbag_directories_compact)

For beginners, the expanded loop is usually better for teaching. The compact version is useful after students understand the underlying steps.

## 15. Sorting ROS bag directories

In [ ]:
sorted_bags = sorted(rosbag_directories)

for index, bag_name in enumerate(sorted_bags):
    print(index, bag_name)

## 16. Introducing NumPy

Python lists are excellent general containers, but numerical robotics calculations are usually better handled with NumPy arrays.

In [ ]:
import numpy as np

joint_position_list = [0.1, 0.5, -0.2]
joint_position_array = np.array(joint_position_list)

print(joint_position_list)
print(joint_position_array)

print(type(joint_position_list))
print(type(joint_position_array))

## 17. List multiplication versus NumPy multiplication

This difference is extremely important.

In [ ]:
python_list = [1, 2, 3]
numpy_array = np.array([1, 2, 3])

print("List multiplied by 2:", python_list * 2)
print("Array multiplied by 2:", numpy_array * 2)

For a Python list, `* 2` repeats the list.

For a NumPy array, `* 2` performs element-by-element multiplication.

## 18. Vector operations

In [ ]:
q = np.array([0.2, 0.4, -0.1])
q_desired = np.array([0.5, 0.3, 0.0])

position_error = q_desired - q

print("q =", q)
print("q_desired =", q_desired)
print("position error =", position_error)

## 19. Matrix operations

In [ ]:
gain_matrix = np.array([
    [10.0, 0.0, 0.0],
    [0.0, 8.0, 0.0],
    [0.0, 0.0, 6.0],
])

control_input = gain_matrix @ position_error

print(control_input)

The `@` operator performs matrix multiplication.

This is much closer to the matrix-oriented operations familiar from MATLAB.

## 20. Converting between lists and arrays

ROS 2 message fields may expect ordinary Python sequences, while calculations may be easier in NumPy.

In [ ]:
message_position = [0.1, 0.2, 0.3]

q_array = np.array(message_position, dtype=float)
q_array = q_array + 0.05

updated_message_position = q_array.tolist()

print(updated_message_position)
print(type(updated_message_position))

A common ROS 2 pattern is:

```text
message field -> NumPy array -> calculation -> Python list -> message field
```

## 21. ROS 2–style message data without requiring ROS 2

The following dictionary imitates the structure of a joint-state message.

In [ ]:
joint_state_message = {
    "name": ["joint_1", "joint_2", "joint_3"],
    "position": [0.2, -0.1, 0.5],
    "velocity": [0.0, 0.1, -0.1],
}

print(joint_state_message["name"])
print(joint_state_message["position"])

The joint names remain a Python list of strings.

The joint positions can be converted into a NumPy array for calculations.

In [ ]:
joint_names = joint_state_message["name"]
joint_positions = np.array(joint_state_message["position"], dtype=float)

for joint_name, joint_position in zip(joint_names, joint_positions):
    print(f"{joint_name}: {joint_position:.3f} rad")

## 22. Why `zip` is useful

`zip` lets us iterate through corresponding elements from multiple lists or arrays.

In [ ]:
joint_names = ["joint_1", "joint_2", "joint_3"]
joint_positions = [0.1, 0.2, 0.3]
joint_velocities = [0.0, -0.1, 0.1]

for name, position, velocity in zip(
    joint_names,
    joint_positions,
    joint_velocities,
):
    print(name, position, velocity)

## 23. A simple callback-style function

ROS 2 subscriber callbacks receive a message object and process its fields.

This function imitates that workflow.

In [ ]:
def process_joint_state(message):
    names = message["name"]
    positions = np.array(message["position"], dtype=float)

    if len(names) != len(positions):
        raise ValueError("Joint names and positions must have the same length.")

    for name, position in zip(names, positions):
        print(f"{name}: {position:.3f} rad")

    return positions


q_received = process_joint_state(joint_state_message)
print("Returned NumPy array:", q_received)

## 24. Optional ROS 2 import test

This cell checks whether the notebook is running inside a ROS 2 Python environment.

In [ ]:
try:
    import rclpy
    from sensor_msgs.msg import JointState

    print("ROS 2 Python packages are available.")
except ImportError:
    rclpy = None
    JointState = None

    print("ROS 2 Python packages are not available in this environment.")
    print("The earlier notebook sections can still be completed normally.")

## 25. Minimal ROS 2 subscriber example

The following code is provided as a reference. It is only defined when ROS 2 packages are available.

It demonstrates:

- importing ROS 2 packages,
- defining a node class,
- creating a subscription,
- reading list-like message fields,
- converting numerical data into a NumPy array,
- iterating with `zip`.

In [ ]:
if rclpy is not None:
    from rclpy.node import Node

    class JointStateTeachingNode(Node):
        def __init__(self):
            super().__init__("joint_state_teaching_node")

            self.subscription = self.create_subscription(
                JointState,
                "/joint_states",
                self.joint_state_callback,
                10,
            )

        def joint_state_callback(self, message):
            joint_names = list(message.name)
            joint_positions = np.array(message.position, dtype=float)

            if len(joint_names) != len(joint_positions):
                self.get_logger().warning(
                    "Joint name and position lengths do not match."
                )
                return

            for name, position in zip(joint_names, joint_positions):
                self.get_logger().info(
                    f"{name}: {position:.3f} rad"
                )
else:
    print("ROS 2 is unavailable, so the node class was not created.")

## 26. Complete ROS 2 execution pattern

A normal ROS 2 Python node uses the following structure:

```python
def main(args=None):
    rclpy.init(args=args)

    node = JointStateTeachingNode()
    rclpy.spin(node)

    node.destroy_node()
    rclpy.shutdown()


if __name__ == "__main__":
    main()
```

This should normally be saved as a Python file inside a ROS 2 package rather than executed repeatedly inside a notebook.

## 27. Teaching exercise: inspect a ROS bag root directory

Complete the function so that it:

1. checks whether the root directory exists,
2. finds subdirectories beginning with `rosbag2_`,
3. sorts the result,
4. returns a Python list.

In [ ]:
def find_rosbag_directories(root_directory):
    if not os.path.isdir(root_directory):
        raise FileNotFoundError(
            f"Directory does not exist: {root_directory}"
        )

    bag_directories = []

    for entry in os.listdir(root_directory):
        full_path = os.path.join(root_directory, entry)

        if os.path.isdir(full_path) and entry.startswith("rosbag2_"):
            bag_directories.append(entry)

    return sorted(bag_directories)


found_bags = find_rosbag_directories(teaching_root)
print(found_bags)

## 28. Teaching exercise: calculate joint errors

The desired and measured joint positions should have the same shape.

In [ ]:
def calculate_joint_error(desired_position, measured_position):
    desired = np.asarray(desired_position, dtype=float)
    measured = np.asarray(measured_position, dtype=float)

    if desired.shape != measured.shape:
        raise ValueError(
            "Desired and measured positions must have the same shape."
        )

    return desired - measured


desired_q = [0.5, 0.0, -0.2]
measured_q = [0.4, -0.1, -0.1]

error_q = calculate_joint_error(desired_q, measured_q)
print(error_q)

## 29. Recommended mental model

Use a **Python list** when you need:

- names,
- topic strings,
- file names,
- directory entries,
- collections of ROS 2 objects,
- mixed data types,
- data that grows with `append()`.

Use a **NumPy array** when you need:

- vectors,
- matrices,
- joint positions,
- velocities,
- accelerations,
- sensor samples,
- control calculations,
- linear algebra.

Use an **object or message** when data has named fields and a defined interface.

Typical ROS 2 flow:

```text
operating system
    -> list of files or directories

ROS 2 message
    -> named fields
    -> some fields contain Python sequences

numerical field
    -> NumPy array
    -> calculation
    -> Python list
    -> outgoing ROS 2 message
```

## 30. Final review questions

1. Why does `data[2:4]` exclude index 4?
2. When is direct list iteration better than `range(len(...))`?
3. What does `enumerate()` provide?
4. Why should paths be built with `os.path.join()`?
5. What type does `os.listdir()` return?
6. Why does `[1, 2, 3] * 2` behave differently from `np.array([1, 2, 3]) * 2`?
7. Why might a ROS 2 message field be converted into a NumPy array?
8. Why is `zip()` useful for joint names and joint positions?
9. Which parts of a ROS 2 node are best kept as lists?
10. Which parts are best converted into arrays?